# Phase I — Unsupervised Structural Analysis
## Bank Marketing Dataset — ML Lab Project (UPC-FIB, MDS 2026)

**Authors:** Arman Bazarchi, Ines Maria Madeira Prates

---

**Prerequisites:** Run `preprocessing.ipynb` first to generate the processed data in `data/`.

### Outline
1. Load Processed Data
2. PCA — Principal Component Analysis
3. Clustering — K-Means & GMM
4. Cluster–Target Alignment Analysis
5. Summary & Discussion

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

---
## 1. Load Processed Data

In [ ]:
X_scaled_df = pd.read_csv('data/X_scaled.csv')
y = pd.read_csv('data/y.csv')['y']
df_processed = pd.read_csv('data/df_processed.csv')
df_raw = pd.read_csv('data/df_raw.csv')
feature_names = X_scaled_df.columns.tolist()

X_scaled = X_scaled_df.values

print(f"Feature matrix: {X_scaled.shape}")
print(f"Target: {y.shape[0]} samples, {y.sum()} positive ({y.mean()*100:.1f}%)")
print(f"Number of features: {len(feature_names)}")

---
## 2. PCA — Principal Component Analysis

### 2.1 Full PCA — Explained Variance Analysis

In [ ]:
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scree plot
axes[0].bar(range(1, len(explained_var) + 1), explained_var, alpha=0.7,
            color='steelblue', edgecolor='black')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')
axes[0].set_xlim(0, 30)

# Cumulative variance
axes[1].plot(range(1, len(cumulative_var) + 1), cumulative_var, 'o-',
             color='steelblue', markersize=4)
axes[1].axhline(y=0.90, color='red', linestyle='--', label='90% threshold')
axes[1].axhline(y=0.95, color='orange', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
axes[1].set_xlim(0, 40)

plt.suptitle('PCA — Explained Variance Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

n_90 = np.argmax(cumulative_var >= 0.90) + 1
n_95 = np.argmax(cumulative_var >= 0.95) + 1
print(f"Components for 90% variance: {n_90}")
print(f"Components for 95% variance: {n_95}")
print(f"Total features: {X_scaled.shape[1]}")

### 2.2 Top Principal Components — Feature Loadings

In [ ]:
n_top = 5
loadings = pd.DataFrame(
    pca_full.components_[:n_top].T,
    columns=[f'PC{i+1}' for i in range(n_top)],
    index=feature_names
)

fig, axes = plt.subplots(1, n_top, figsize=(25, 8))
for i in range(n_top):
    pc_col = f'PC{i+1}'
    top_features = loadings[pc_col].abs().nlargest(10)
    top_loadings = loadings.loc[top_features.index, pc_col]

    colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in top_loadings.values]
    axes[i].barh(range(len(top_loadings)), top_loadings.values, color=colors, edgecolor='black')
    axes[i].set_yticks(range(len(top_loadings)))
    axes[i].set_yticklabels(top_loadings.index, fontsize=9)
    axes[i].set_title(f'{pc_col} ({explained_var[i]*100:.1f}%)', fontweight='bold')
    axes[i].axvline(x=0, color='black', linewidth=0.5)

plt.suptitle('Top 10 Feature Loadings per Principal Component', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.3 2D PCA Projection — Colored by Target

In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
for label, color, name in [(0, '#e74c3c', 'No'), (1, '#2ecc71', 'Yes')]:
    mask = y == label
    plt.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
                c=color, label=name, alpha=0.3, s=10, edgecolors='none')

plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('2D PCA Projection — Colored by Target (y)', fontweight='bold')
plt.legend(title='Subscribed', markerscale=3)
plt.tight_layout()
plt.show()

---
## 3. Clustering — K-Means & Gaussian Mixture Models

### 3.1 Determining the Number of Clusters

We use three criteria:
- **Elbow Method** (inertia / within-cluster sum of squares)
- **Silhouette Score** (cohesion vs. separation)
- **BIC** (Bayesian Information Criterion, for GMM)

We cluster on PCA-reduced data (retaining 90% variance) to reduce noise and computation.

In [ ]:
n_components_cluster = n_90
pca_cluster = PCA(n_components=n_components_cluster, random_state=42)
X_pca = pca_cluster.fit_transform(X_scaled)

print(f"Using {n_components_cluster} PCA components for clustering (90% variance retained)")
print(f"PCA-reduced data shape: {X_pca.shape}")

In [ ]:
K_range = range(2, 11)

inertias = []
silhouette_scores_km = []
bic_scores = []
silhouette_scores_gmm = []

for k in K_range:
    # K-Means
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouette_scores_km.append(silhouette_score(X_pca, km_labels))

    # GMM
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=5)
    gmm_labels = gmm.fit_predict(X_pca)
    bic_scores.append(gmm.bic(X_pca))
    silhouette_scores_gmm.append(silhouette_score(X_pca, gmm_labels))

    print(f"k={k:2d} | KM Silhouette: {silhouette_scores_km[-1]:.4f} | "
          f"GMM Silhouette: {silhouette_scores_gmm[-1]:.4f} | GMM BIC: {bic_scores[-1]:.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Elbow
axes[0].plot(K_range, inertias, 'o-', color='steelblue', markersize=6)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method (K-Means)', fontweight='bold')
axes[0].set_xticks(list(K_range))

# Silhouette
axes[1].plot(K_range, silhouette_scores_km, 'o-', color='steelblue',
             label='K-Means', markersize=6)
axes[1].plot(K_range, silhouette_scores_gmm, 's--', color='darkorange',
             label='GMM', markersize=6)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score Comparison', fontweight='bold')
axes[1].legend()
axes[1].set_xticks(list(K_range))

# BIC
axes[2].plot(K_range, bic_scores, 's-', color='darkorange', markersize=6)
axes[2].set_xlabel('Number of Components (k)')
axes[2].set_ylabel('BIC')
axes[2].set_title('BIC (Gaussian Mixture)', fontweight='bold')
axes[2].set_xticks(list(K_range))

plt.suptitle('Cluster Number Selection', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.2 K-Means Clustering (Best k)

In [ ]:
best_k_km = list(K_range)[np.argmax(silhouette_scores_km)]
print(f"Best k by silhouette score (K-Means): {best_k_km}")

km_final = KMeans(n_clusters=best_k_km, random_state=42, n_init=10)
km_labels = km_final.fit_predict(X_pca)

print(f"\nCluster sizes (K-Means, k={best_k_km}):")
for c in range(best_k_km):
    print(f"  Cluster {c}: {(km_labels == c).sum()} samples")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 2D scatter
scatter = axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=km_labels,
                          cmap='Set2', alpha=0.3, s=10, edgecolors='none')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title(f'K-Means Clusters (k={best_k_km}) in PCA Space', fontweight='bold')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Silhouette plot
sil_samples = silhouette_samples(X_pca, km_labels)
y_lower = 10
cmap = plt.cm.Set2
for c in range(best_k_km):
    cluster_sil = sil_samples[km_labels == c]
    cluster_sil.sort()
    y_upper = y_lower + len(cluster_sil)
    color = cmap(c / best_k_km)
    axes[1].fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                          facecolor=color, edgecolor=color, alpha=0.7)
    axes[1].text(-0.05, y_lower + 0.5 * len(cluster_sil), str(c), fontweight='bold')
    y_lower = y_upper + 10

avg_sil = silhouette_score(X_pca, km_labels)
axes[1].axvline(x=avg_sil, color='red', linestyle='--', label=f'Mean: {avg_sil:.3f}')
axes[1].set_xlabel('Silhouette Coefficient')
axes[1].set_ylabel('Cluster')
axes[1].set_title('Silhouette Plot (K-Means)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.3 Gaussian Mixture Model (GMM)

In [ ]:
best_k_gmm = list(K_range)[np.argmin(bic_scores)]
print(f"Best k by BIC (GMM): {best_k_gmm}")

gmm_final = GaussianMixture(n_components=best_k_gmm, random_state=42, n_init=5)
gmm_labels = gmm_final.fit_predict(X_pca)
gmm_probs = gmm_final.predict_proba(X_pca)

print(f"\nCluster sizes (GMM, k={best_k_gmm}):")
for c in range(best_k_gmm):
    print(f"  Component {c}: {(gmm_labels == c).sum()} samples")

print(f"\nGMM Silhouette Score: {silhouette_score(X_pca, gmm_labels):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# GMM clusters
scatter = axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=gmm_labels,
                          cmap='Set2', alpha=0.3, s=10, edgecolors='none')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title(f'GMM Components (k={best_k_gmm}) in PCA Space', fontweight='bold')
plt.colorbar(scatter, ax=axes[0], label='Component')

# Assignment confidence
max_probs = gmm_probs.max(axis=1)
scatter2 = axes[1].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=max_probs,
                           cmap='RdYlGn', alpha=0.3, s=10, edgecolors='none')
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('GMM Assignment Confidence', fontweight='bold')
plt.colorbar(scatter2, ax=axes[1], label='Max Posterior Probability')

plt.tight_layout()
plt.show()

---
## 4. Cluster–Target Alignment Analysis

We examine whether the discovered clusters align with the target variable — i.e., do some clusters have higher subscription rates than others?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means
km_target_ct = pd.crosstab(km_labels, y, normalize='index') * 100
km_target_ct.columns = ['No', 'Yes']
km_target_ct.plot(kind='bar', stacked=True, ax=axes[0],
                  color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title(f'K-Means (k={best_k_km}) — Target per Cluster', fontweight='bold')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Percentage')
axes[0].legend(title='Subscribed')

# GMM
gmm_target_ct = pd.crosstab(gmm_labels, y, normalize='index') * 100
gmm_target_ct.columns = ['No', 'Yes']
gmm_target_ct.plot(kind='bar', stacked=True, ax=axes[1],
                   color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[1].set_title(f'GMM (k={best_k_gmm}) — Target per Component', fontweight='bold')
axes[1].set_xlabel('Component')
axes[1].set_ylabel('Percentage')
axes[1].legend(title='Subscribed')

plt.suptitle('Cluster–Target Alignment', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print("K-Means — Target distribution per cluster:")
print("=" * 55)
km_detail = pd.crosstab(km_labels, y)
km_detail.columns = ['No', 'Yes']
km_detail['Total'] = km_detail.sum(axis=1)
km_detail['Yes_Rate (%)'] = (km_detail['Yes'] / km_detail['Total'] * 100).round(2)
print(km_detail)
print(f"\nOverall subscription rate: {y.mean()*100:.2f}%")

In [ ]:
print("GMM — Target distribution per component:")
print("=" * 55)
gmm_detail = pd.crosstab(gmm_labels, y)
gmm_detail.columns = ['No', 'Yes']
gmm_detail['Total'] = gmm_detail.sum(axis=1)
gmm_detail['Yes_Rate (%)'] = (gmm_detail['Yes'] / gmm_detail['Total'] * 100).round(2)
print(gmm_detail)

### 4.1 Cluster Profiling — Feature Means per Cluster

In [ ]:
# Profile clusters using original numerical features for interpretability
numerical_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()

df_profile = df_raw[numerical_cols].copy()
df_profile['km_cluster'] = km_labels

print("K-Means Cluster Profiles (mean of numerical features):")
print("=" * 80)
km_profile = df_profile.groupby('km_cluster')[numerical_cols].mean().round(2)
print(km_profile.T.to_string())

In [ ]:
# Heatmap of standardized cluster means
km_profile_z = (km_profile - km_profile.mean()) / (km_profile.std() + 1e-10)

plt.figure(figsize=(14, 8))
sns.heatmap(km_profile_z.T, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5)
plt.title('K-Means Cluster Profiles (Standardized Feature Means)', fontweight='bold')
plt.xlabel('Cluster')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

---
## 5. Summary & Discussion

### Key Findings — Phase I

**PCA Analysis:**
- After one-hot encoding, the feature space expands significantly.
- The explained variance curve shows gradual accumulation — the data is not low-dimensional.
- The first 2 PCs do not clearly separate the two classes, suggesting the classification boundary is complex and may require non-linear models.

**Clustering:**
- Both K-Means and GMM were evaluated for k=2..10.
- Silhouette scores and cluster–target alignment tables reveal whether the data's natural groupings relate to the subscription outcome.
- Cluster profiling helps identify which client segments are most/least likely to subscribe.

**Implications for Phase II:**
- The lack of clean cluster–target separation suggests supervised models will be essential.
- Class imbalance needs to be addressed (stratified splitting, appropriate metrics: F1, ROC-AUC, AUPR).
- Feature selection may help reduce noise from the expanded feature space.
- The `duration` feature must be excluded from supervised models (data leakage).

---

*Phase II will focus on Linear and Regularized Models (Logistic Regression with Ridge/Lasso penalties).*